In [3]:
import pandas as pd
import numpy as np

# Step 1: Load data
df = pd.read_csv('churnguard_data.csv')

# Step 2: Drop customerID
df = df.drop('customerID', axis=1)

# Step 3: Remove duplicates
df = df.drop_duplicates()

# Step 4: Strip whitespace
df['gender'] = df['gender'].str.strip()
df['PaymentMethod'] = df['PaymentMethod'].str.strip()

# Step 5: Standardise casing
df['Churn'] = df['Churn'].str.strip().str.title()
df['PhoneService'] = df['PhoneService'].str.strip().str.title()
df['PaperlessBilling'] = df['PaperlessBilling'].str.strip().str.title()

# Step 6: Fix Contract
contract_map = {
    'Month-to-month': 'Month-to-month',
    'month to month': 'Month-to-month',
    'month-to-month': 'Month-to-month',
    'Monthly':        'Month-to-month',
    'One year':       'One year',
    'One Year':       'One year',
    'one year':       'One year',
    '1 year':         'One year',
    'Two year':       'Two year',
    'Two Year':       'Two year',
    'two year':       'Two year',
    '2 year':         'Two year',
}
df['Contract'] = df['Contract'].map(contract_map)

# Step 7: Fix InternetService
internet_map = {
    'DSL':         'DSL',
    'dsl':         'DSL',
    'Fiber optic': 'Fiber optic',
    'Fibre optic': 'Fiber optic',
    'FiberOptic':  'Fiber optic',
    'fiber optic': 'Fiber optic',
    'No':          'No',
    'None':        'No',
    'none':        'No',
}
df['InternetService'] = df['InternetService'].map(internet_map)

# Step 8: Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Step 9: Remove impossible tenure
df = df[df['tenure'] > 0]

# Step 10: Remove MonthlyCharges outliers
df = df[(df['MonthlyCharges'] >= 10) & (df['MonthlyCharges'] <= 200)]

# Step 11: Fill missing values
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['tenure'] = df['tenure'].fillna(int(df['tenure'].median()))

# Step 12: Print results
print("Cleaned shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

Cleaned shape: (867, 11)

Missing values:
gender               0
SeniorCitizen        0
tenure               0
PhoneService         0
InternetService     61
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges         0
Churn                0
dtype: int64


# Task 2 — Data Cleaning (ChurnGuard)

## Goal
Clean the raw dataset and produce a ready-to-use DataFrame for modelling.

## Cleaning Steps Applied (in order)

### 1. Drop customerID
```python
df = df.drop('customerID', axis=1)
```
Not useful for prediction — just a unique identifier.

### 2. Remove Duplicates
```python
df = df.drop_duplicates()
```
30 duplicate rows removed.

### 3. Strip Whitespace
```python
df['gender'] = df['gender'].str.strip()
df['PaymentMethod'] = df['PaymentMethod'].str.strip()
```
Extra spaces cause mismatches in grouping and encoding.

### 4. Standardise Casing
```python
df['Churn'] = df['Churn'].str.strip().str.title()
df['PhoneService'] = df['PhoneService'].str.strip().str.title()
df['PaperlessBilling'] = df['PaperlessBilling'].str.strip().str.title()
```
Converts yes/YES/yEs → Yes, no/NO/nO → No

### 5. Fix Contract Column
Mapped 12 inconsistent values → 3 clean values:
Month-to-month = month to month = monthly = month-to-month
One year       = One Year = one year = 1 year
Two year       = Two Year = two year = 2 year

### 6. Fix InternetService Column
Mapped inconsistent values → 3 clean values:
DSL         = dsl
Fiber optic = Fibre optic = FiberOptic = fiber optic
No          = None = none

### 7. Fix TotalCharges — Convert to Numeric
```python
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
```
Was stored as text — junk values (N/A, --, ?) become NaN automatically.

### 8. Remove Impossible tenure Values
```python
df = df[df['tenure'] > 0]
```
Negative and zero tenure is impossible in real world.

### 9. Remove MonthlyCharges Outliers
```python
df = df[(df['MonthlyCharges'] >= 10) & (df['MonthlyCharges'] <= 200)]
```
Values like 0.01, 999, 1500 are clearly wrong entries.

### 10. Fill Missing Values
```python
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges']   = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['tenure']         = df['tenure'].fillna(int(df['tenure'].median()))
```
- MonthlyCharges → mean (continuous, symmetric)
- TotalCharges   → mean (continuous)
- tenure         → median (robust to skew, integer rounded)

## Output
Cleaned shape: (867, 11)
InternetService: 61 missing values remaining
(all other columns: 0 missing)

## Key Learnings
- Real world data always needs cleaning before modelling
- str.strip() removes invisible whitespace issues
- str.title() standardises inconsistent casing
- map() is the cleanest way to fix typos and variants
- pd.to_numeric(errors='coerce') converts junk to NaN safely
- Always remove impossible values before filling missing ones
- Mean for continuous features, Median for skewed or integer features